# Fundamental Solution Source Placement

**Sweep:** `fs_placement`  
**Question:** How far from the boundary should fundamental-solution sources be placed? Does distributing them proportionally to segment length (vs uniformly) matter?

**Sweep variables:**
- `fs_d_scale` ∈ {0.05, 0.1, 0.15, 0.2, 0.3, 0.5} — source distance as a fraction of the inradius
- `fs_seg_strategy` ∈ {length_weighted, uniform}

**Fixed:** `n_fb = 0`, `n_fs = 100`, `fs_d_strategy = 'inradius_fraction'`, `rtol = 1e-12`

**Domains:**
| Domain | Reference eigenvalues |
|---|---|
| disk | ✓ |
| rect (2×1) | ✓ |
| GWW1 | ✓ |
| chevron | ✗ — tension proxy used |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nb_utils

nb_utils.set_publication_style()
RESULTS_DIR = os.path.abspath('../results')

df, raw = nb_utils.load_sweep('fs_placement', RESULTS_DIR)
print(f'Loaded {len(df)} results')
print('Domains:', df['domain_name'].unique())
print('d_scale values:', sorted(df['fs_d_scale'].unique()))
print('Segment strategies:', df['fs_seg_strategy'].unique())
df.head(3)

## Plot 1: Accuracy vs Source Distance

For each domain, how does accuracy vary with source standoff distance? Solid = length-weighted, dashed = uniform distribution.

In [ ]:
all_domains = sorted(df['domain_name'].unique())
colors = nb_utils.domain_color_map(all_domains)
d_scales = sorted(df['fs_d_scale'].unique())
seg_styles = {'length_weighted': '-', 'uniform': '--'}

nrows, ncols = 2, 2
fig, axes = plt.subplots(nrows, ncols, figsize=(10, 7))
axes = axes.flatten()

for idx, dom in enumerate(all_domains[:4]):
    ax = axes[idx]
    has_ref = dom in nb_utils.DOMAINS_WITH_REFERENCE
    metric = 'max_rel_error' if has_ref else 'max_tension'
    ylabel = 'Max relative error' if has_ref else 'Max tension (proxy)'

    for seg_strat, ls in seg_styles.items():
        d = df[(df['domain_name'] == dom) &
               (df['fs_seg_strategy'] == seg_strat)].sort_values('fs_d_scale')
        if d[metric].notna().any():
            ax.semilogy(d['fs_d_scale'], d[metric],
                        marker='o', markersize=5, linestyle=ls,
                        color=colors[dom], label=seg_strat.replace('_', ' '))

    title = nb_utils.label_domain(dom)
    if not has_ref:
        title += ' (tension proxy)'
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Source distance (× inradius)', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(fontsize=8)

fig.suptitle('Accuracy vs FS source standoff distance', fontsize=11)
plt.tight_layout()
plt.show()

## Plot 2: MPS Tension vs Source Distance

Tension is available for all domains. Does it track relative error for domains with reference values?

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(10, 7))
axes = axes.flatten()

for idx, dom in enumerate(all_domains[:4]):
    ax = axes[idx]
    for seg_strat, ls in seg_styles.items():
        d = df[(df['domain_name'] == dom) &
               (df['fs_seg_strategy'] == seg_strat)].sort_values('fs_d_scale')
        ax.semilogy(d['fs_d_scale'], d['median_tension'],
                    marker='o', markersize=5, linestyle=ls,
                    color=colors[dom], label=seg_strat.replace('_', ' '))
    ax.set_title(nb_utils.label_domain(dom), fontsize=9)
    ax.set_xlabel('Source distance (× inradius)', fontsize=9)
    ax.set_ylabel('Median MPS tension', fontsize=9)
    ax.legend(fontsize=8)

fig.suptitle('MPS tension vs FS source standoff distance (all domains)', fontsize=11)
plt.tight_layout()
plt.show()

## Plot 3: Segment Strategy Comparison

Across all d_scale values, which segment distribution strategy achieves the best accuracy?

In [ ]:
seg_strats = list(seg_styles.keys())
x = np.arange(len(all_domains))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
for i, seg_strat in enumerate(seg_strats):
    vals = []
    for dom in all_domains:
        has_ref = dom in nb_utils.DOMAINS_WITH_REFERENCE
        metric = 'max_rel_error' if has_ref else 'max_tension'
        sub = df[(df['domain_name'] == dom) & (df['fs_seg_strategy'] == seg_strat)]
        best = sub[metric].min() if sub[metric].notna().any() else np.nan
        vals.append(best)
    offset = (i - 0.5) * width
    ax.bar(x + offset, vals, width=width * 0.9,
           color=['steelblue', 'coral'][i],
           label=seg_strat.replace('_', ' '))

ax.set_yscale('log')
ax.set_xticks(x)
xlabels = [nb_utils.label_domain(d) + ('' if d in nb_utils.DOMAINS_WITH_REFERENCE else '\n(tension)')
           for d in all_domains]
ax.set_xticklabels(xlabels, fontsize=9)
ax.set_ylabel('Best metric (min over all d_scale)')
ax.set_title('Segment distribution strategy: best achievable accuracy')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Plot 4: Per-Eigenvalue Errors vs Source Distance

For reference domains, using length-weighted distribution: do all eigenvalues respond similarly to source distance?

In [ ]:
ref_domains = [d for d in all_domains if d in nb_utils.DOMAINS_WITH_REFERENCE]

# index: (domain, fs_d_scale, fs_seg_strategy) → raw result
index = {}
for r in raw:
    cfg = r.config
    key = (cfg.domain_name, round(cfg.fs_d_scale, 4), cfg.fs_seg_strategy)
    index[key] = r

d_scale_cmap = plt.cm.get_cmap('viridis', len(d_scales))
d_scale_colors = {d: d_scale_cmap(i / (len(d_scales) - 1)) for i, d in enumerate(d_scales)}

fig, axes = plt.subplots(1, len(ref_domains), figsize=(4.5 * len(ref_domains), 4))
if len(ref_domains) == 1:
    axes = [axes]

for ax, dom in zip(axes, ref_domains):
    for d_sc in d_scales:
        r = index.get((dom, round(d_sc, 4), 'length_weighted'))
        if r is None or r.rel_errors is None:
            continue
        eig_idx = np.arange(1, len(r.rel_errors) + 1)
        ax.semilogy(eig_idx, r.rel_errors,
                    marker='o', markersize=4,
                    color=d_scale_colors[d_sc],
                    label=f'd={d_sc:.2f}')
    ax.set_title(nb_utils.label_domain(dom), fontsize=9)
    ax.set_xlabel('Eigenvalue index', fontsize=9)
    ax.set_ylabel('Relative error', fontsize=9)
    ax.legend(fontsize=7)

fig.suptitle('Per-eigenvalue errors vs source distance (length-weighted)', fontsize=11)
plt.tight_layout()
plt.show()

## Plot 5: Wall Time vs Source Distance

Source placement changes the geometry but not the solver size — wall time should be nearly constant.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for dom in all_domains:
    d = df[(df['domain_name'] == dom) &
           (df['fs_seg_strategy'] == 'length_weighted')].sort_values('fs_d_scale')
    ax.plot(d['fs_d_scale'], d['wall_time'],
            marker='o', color=colors[dom],
            label=nb_utils.label_domain(dom))

ax.set_xlabel('Source distance (× inradius)')
ax.set_ylabel('Wall time (s)')
ax.set_title('Computation cost vs source distance (length-weighted)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()